# Compressed Models, Compressed Self-Knowledge? — Run-All Experiment

**What this does (no editing needed):**
1. Samples 10 answers per question from Qwen2.5-1.5B-Instruct (FP16) on 1,000 TriviaQA questions.
2. Labels each answer correct/wrong against TriviaQA aliases.
3. Feeds the *same* question+answer text through the model at **FP16, INT8 and NF4** and caches hidden states (teacher-forced design).
4. Trains linear probes and reports **pooled**, **within-question** and **question-only** AUROC, cross-precision transfer, best-of-N, and bootstrap CIs.

**How to run on Kaggle:** Settings → Accelerator **GPU T4 x1**, Internet **On** → *Run All*.
Expected time: roughly 1.5–3 hours (estimate). When it finishes, download `/kaggle/working/results.zip` and send it back.

Only change the CONFIG cell if you want a different model or size.

In [ ]:
!pip install -q -U bitsandbytes

In [ ]:
# ============ CONFIG ============
MODEL_NAME   = "Qwen/Qwen2.5-1.5B-Instruct"   # not gated, no HF token needed
N_QUESTIONS  = 1000
K            = 10          # sampled answers per question
TEMPERATURE  = 0.7
TOP_P        = 0.95
MAX_NEW_TOK  = 24
GEN_BATCH_Q  = 8           # questions per generation batch (lower if out of memory)
LAYER_STEP   = 4           # store every 4th layer
PRECISIONS   = ["fp16", "int8", "nf4"]
SEED         = 0
N_BOOT       = 1000
OUT_DIR      = "/kaggle/working" if __import__("os").path.exists("/kaggle/working") else "./outputs"
# ================================
import os, json, re, string, gc, time, random, zipfile
import numpy as np, pandas as pd, torch
from collections import Counter
os.makedirs(OUT_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 1. Data and answer normalization

In [ ]:
from datasets import load_dataset
ds = load_dataset("mandarjoshi/trivia_qa", "rc.nocontext", split="validation")
idx = np.random.RandomState(SEED).permutation(len(ds))[:N_QUESTIONS]
questions = [ds[int(i)] for i in idx]

def normalize(s):
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def is_correct(pred, aliases):
    p = normalize(pred)
    if not p:
        return 0
    for a in aliases:
        a = normalize(a)
        if a and (p == a or re.search(r"\b" + re.escape(a) + r"\b", p)):
            return 1
    return 0

def clean_answer(text):
    text = text.strip().split("\n")[0].strip()
    return text

qdata = []
for qi, ex in enumerate(questions):
    aliases = list(set(ex["answer"]["aliases"] + ex["answer"]["normalized_aliases"] + [ex["answer"]["value"]]))
    qdata.append({"qid": qi, "question": ex["question"], "aliases": aliases})
print(len(qdata), qdata[0]["question"])

## 2. Model loading helpers

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def load_model(precision):
    if precision == "fp16":
        m = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="cuda")
    elif precision == "int8":
        m = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="cuda", torch_dtype=torch.float16,
              quantization_config=BitsAndBytesConfig(load_in_8bit=True))
    elif precision == "nf4":
        m = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="cuda", torch_dtype=torch.float16,
              quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                     bnb_4bit_use_double_quant=True,
                                                     bnb_4bit_compute_dtype=torch.float16))
    else:
        raise ValueError(precision)
    m.eval()
    return m

def free(m):
    del m; gc.collect(); torch.cuda.empty_cache()

def prompt_text(q):
    msgs = [{"role": "system", "content": "Answer the question with a short answer only (a few words). No explanation."},
            {"role": "user", "content": q}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

## 3. Generate answers with the FP16 model (greedy + K samples)

In [ ]:
@torch.no_grad()
def generate(model, prompts, do_sample, n):
    tok.padding_side = "left"
    enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
    out = model.generate(**enc, max_new_tokens=MAX_NEW_TOK, do_sample=do_sample,
                         temperature=TEMPERATURE if do_sample else None,
                         top_p=TOP_P if do_sample else None, top_k=None,
                         num_return_sequences=n, pad_token_id=tok.pad_token_id)
    new = out[:, enc["input_ids"].shape[1]:]
    texts = tok.batch_decode(new, skip_special_tokens=True)
    return [texts[i*n:(i+1)*n] for i in range(len(prompts))]

def greedy_accuracy(model):
    hits = []
    for b in range(0, len(qdata), GEN_BATCH_Q*2):
        batch = qdata[b:b+GEN_BATCH_Q*2]
        outs = generate(model, [prompt_text(d["question"]) for d in batch], False, 1)
        for d, o in zip(batch, outs):
            hits.append(is_correct(clean_answer(o[0]), d["aliases"]))
    return float(np.mean(hits))

t0 = time.time()
model = load_model("fp16")
rows = []
for b in range(0, len(qdata), GEN_BATCH_Q):
    batch = qdata[b:b+GEN_BATCH_Q]
    prompts = [prompt_text(d["question"]) for d in batch]
    greedy = generate(model, prompts, False, 1)
    samples = generate(model, prompts, True, K)
    for d, g, ss in zip(batch, greedy, samples):
        g = clean_answer(g[0])
        rows.append({"qid": d["qid"], "sample": -1, "answer": g, "label": is_correct(g, d["aliases"])})
        for k, s in enumerate(ss):
            s = clean_answer(s)
            rows.append({"qid": d["qid"], "sample": k, "answer": s, "label": is_correct(s, d["aliases"])})
    if b % (GEN_BATCH_Q*10) == 0:
        print(f"{b}/{len(qdata)}  {time.time()-t0:.0f}s")
ans = pd.DataFrame(rows)
ans = ans[ans.answer.str.len() > 0].reset_index(drop=True)
ans.to_csv(f"{OUT_DIR}/answers.csv", index=False)
greedy_acc = {"fp16": float(ans[ans["sample"]==-1].label.mean())}
samp = ans[ans["sample"]>=0]
mixed = samp.groupby("qid").label.agg(lambda x: 0 < x.mean() < 1)
print("generation time (s):", round(time.time()-t0))
print("FP16 greedy acc:", greedy_acc["fp16"], "| sample acc:", round(samp.label.mean(), 3),
      "| mixed-correctness questions:", int(mixed.sum()), "of", len(mixed))

## 4. Teacher-forced extraction at each precision (same text, same labels)

In [ ]:
@torch.no_grad()
def extract(model):
    """For each sampled answer: hidden states at last answer token + mean answer logprob.
    For each question: hidden states at last prompt token."""
    tok.padding_side = "right"
    n_layers = model.config.num_hidden_layers
    layers = list(range(LAYER_STEP, n_layers + 1, LAYER_STEP))
    if n_layers not in layers: layers.append(n_layers)
    A_feat, A_lp, Q_feat = {}, {}, {}
    for qid, grp in samp.groupby("qid"):
        p = prompt_text(qdata[qid]["question"])
        p_ids = tok(p, add_special_tokens=False)["input_ids"]
        P = len(p_ids)
        texts = [p + a for a in grp.answer]
        enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        out = model(**enc, output_hidden_states=True)
        hs = torch.stack([out.hidden_states[l] for l in layers], dim=1)   # [B, L, T, D]
        last = enc["attention_mask"].sum(1) - 1
        logp = torch.log_softmax(out.logits.float(), -1)
        for j, row_id in enumerate(grp.index):
            e = int(last[j])
            A_feat[row_id] = hs[j, :, e, :].float().cpu().numpy().astype(np.float16)
            if e >= P:
                tgt = enc["input_ids"][j, P:e+1]
                lp = logp[j, P-1:e, :].gather(-1, tgt[:, None]).squeeze(-1)
                A_lp[row_id] = float(lp.mean())
            else:
                A_lp[row_id] = float("nan")
        Q_feat[qid] = hs[0, :, P-1, :].float().cpu().numpy().astype(np.float16)
        del out, hs, logp
    order = list(samp.index)
    XA = np.stack([A_feat[i] for i in order])           # [N_ans, L, D]
    LP = np.array([A_lp[i] for i in order])
    XQ = np.stack([Q_feat[q] for q in samp.loc[order, "qid"]])  # question feature per answer row
    return XA, XQ, LP, layers

feats = {}
for prec in PRECISIONS:
    t0 = time.time()
    if prec != "fp16":
        try: free(model)
        except NameError: pass
        model = load_model(prec)
        greedy_acc[prec] = greedy_accuracy(model)
    XA, XQ, LP, layers = extract(model)
    feats[prec] = {"XA": XA, "XQ": XQ, "LP": LP}
    np.save(f"{OUT_DIR}/XA_{prec}.npy", XA)
    print(prec, XA.shape, f"{time.time()-t0:.0f}s", "greedy acc:", greedy_acc.get(prec))
free(model)
LAYERS = layers

## 5. Analysis functions

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

def wq_auc(s, y, q):
    num, den = 0.0, 0
    df = pd.DataFrame({"s": s, "y": y, "q": q})
    for _, g in df.groupby("q"):
        pos, neg = g.s[g.y == 1].values, g.s[g.y == 0].values
        if len(pos) and len(neg):
            d = pos[:, None] - neg[None, :]
            num += (d > 0).sum() + 0.5 * (d == 0).sum()
            den += d.size
    return num / den if den else float("nan")

def pooled_auc(s, y):
    return roc_auc_score(y, s) if 0 < y.mean() < 1 else float("nan")

def fit_probe(X, y, Xv, yv):
    best, best_auc = None, -1
    for C in [0.001, 0.01, 0.1, 1.0]:
        clf = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=2000))
        clf.fit(X, y)
        a = pooled_auc(clf.decision_function(Xv), yv)
        if a > best_auc: best, best_auc = clf, a
    return best, best_auc

def majority_vote_acc(df):
    accs = []
    for _, g in df.groupby("qid"):
        norm = g.answer.map(normalize)
        top = Counter(norm).most_common(1)[0][0]
        accs.append(g.label.values[(norm == top).values][0])
    return float(np.mean(accs))

def select_acc(df, score_col):
    return float(df.loc[df.groupby("qid")[score_col].idxmax(), "label"].mean())

## 6. Question-grouped split, probes per precision, transfer, best-of-N

In [ ]:
y = samp.label.values
qids = samp.qid.values
uq = np.unique(qids)
rng = np.random.RandomState(SEED); rng.shuffle(uq)
n = len(uq)
tr_q, va_q, te_q = set(uq[:int(.6*n)]), set(uq[int(.6*n):int(.8*n)]), set(uq[int(.8*n):])
tr = np.isin(qids, list(tr_q)); va = np.isin(qids, list(va_q)); te = np.isin(qids, list(te_q))

# sampling-agreement baseline (text only, same for all precisions)
norm = samp.answer.map(normalize).values
agree = np.zeros(len(samp))
for q in uq:
    m = np.where(qids == q)[0]
    c = Counter(norm[m])
    for i in m: agree[i] = (c[norm[i]] - 1) / max(len(m) - 1, 1)

results = {"model": MODEL_NAME, "n_questions": N_QUESTIONS, "K": K, "layers": LAYERS,
           "greedy_acc": greedy_acc, "sample_acc": float(y.mean()),
           "n_mixed_questions_total": int(mixed.sum()),
           "n_test_questions": len(te_q),
           "n_test_mixed": int(mixed[mixed.index.isin(te_q)].sum()),
           "per_layer": {}, "main": {}, "transfer": {}, "best_of_n": {}}

probes, test_scores = {}, {}
for prec in PRECISIONS:
    XA, XQ, LP = feats[prec]["XA"].astype(np.float32), feats[prec]["XQ"].astype(np.float32), feats[prec]["LP"]
    layer_res = {}
    for li, L in enumerate(LAYERS):
        clf, va_auc = fit_probe(XA[tr, li], y[tr], XA[va, li], y[va])
        layer_res[L] = {"val_pooled": va_auc, "clf": clf}
    best_li = int(np.argmax([layer_res[L]["val_pooled"] for L in LAYERS]))
    best_L = LAYERS[best_li]
    clf = layer_res[best_L]["clf"]
    qclf, _ = fit_probe(XQ[tr, best_li], y[tr], XQ[va, best_li], y[va])
    s_ans = clf.decision_function(XA[te, best_li])
    s_q = qclf.decision_function(XQ[te, best_li])
    probes[prec] = (clf, best_li)
    test_scores[prec] = {"probe": s_ans, "qonly": s_q, "logprob": np.nan_to_num(LP[te], nan=-99.0)}
    results["per_layer"][prec] = {int(L): {"val_pooled": layer_res[L]["val_pooled"],
                                           "test_pooled": pooled_auc(layer_res[L]["clf"].decision_function(XA[te, li]), y[te]),
                                           "test_wq": wq_auc(layer_res[L]["clf"].decision_function(XA[te, li]), y[te], qids[te])}
                                  for li, L in enumerate(LAYERS)}
    results["main"][prec] = {"layer": int(best_L),
        "probe_pooled": pooled_auc(s_ans, y[te]), "probe_wq": wq_auc(s_ans, y[te], qids[te]),
        "qonly_pooled": pooled_auc(s_q, y[te]),
        "logprob_pooled": pooled_auc(test_scores[prec]["logprob"], y[te]),
        "logprob_wq": wq_auc(test_scores[prec]["logprob"], y[te], qids[te])}
    print(prec, results["main"][prec])

results["main"]["agreement_baseline"] = {"pooled": pooled_auc(agree[te], y[te]), "wq": wq_auc(agree[te], y[te], qids[te])}

# cross-precision transfer: probe trained at i, applied to precision j features (same layer)
for pi in PRECISIONS:
    clf, li = probes[pi]
    for pj in PRECISIONS:
        s = clf.decision_function(feats[pj]["XA"][te, li].astype(np.float32))
        results["transfer"][f"{pi}->{pj}"] = {"pooled": pooled_auc(s, y[te]), "wq": wq_auc(s, y[te], qids[te])}

# best-of-N on test questions
te_df = samp[te].copy()
for prec in PRECISIONS:
    te_df[f"s_{prec}"] = test_scores[prec]["probe"]
greedy_rows = ans[(ans["sample"] == -1) & ans.qid.isin(te_q)]
results["best_of_n"] = {"greedy": float(greedy_rows.label.mean()),
                        "majority_vote": majority_vote_acc(te_df),
                        "oracle": float(te_df.groupby("qid").label.max().mean()),
                        **{f"probe_select_{p}": select_acc(te_df, f"s_{p}") for p in PRECISIONS}}
te_df.to_csv(f"{OUT_DIR}/test_scores.csv", index=False)
print(json.dumps(results["best_of_n"], indent=1))

## 7. Paired bootstrap over test questions (CIs, differences, retention ratios)

In [ ]:
te_idx = np.where(te)[0]
te_q_arr = np.array(sorted(te_q))
by_q = {q: np.where(qids[te] == q)[0] for q in te_q_arr}
rng = np.random.RandomState(SEED)
boot = {k: [] for k in ["pooled", "wq", "qonly"]}
store = {p: {"pooled": [], "wq": [], "qonly": []} for p in PRECISIONS}
for b in range(N_BOOT):
    samp_q = rng.choice(te_q_arr, size=len(te_q_arr), replace=True)
    rows_b, q_b = [], []
    for j, q in enumerate(samp_q):
        r = by_q[q]; rows_b.append(r); q_b.append(np.full(len(r), j))
    rows_b = np.concatenate(rows_b); q_b = np.concatenate(q_b)
    yb = y[te][rows_b]
    for p in PRECISIONS:
        s = test_scores[p]["probe"][rows_b]
        store[p]["pooled"].append(pooled_auc(s, yb))
        store[p]["wq"].append(wq_auc(s, yb, q_b))
        store[p]["qonly"].append(pooled_auc(test_scores[p]["qonly"][rows_b], yb))

def ci(a):
    a = np.array(a); a = a[~np.isnan(a)]
    return [float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5))]

results["bootstrap"] = {}
for p in PRECISIONS:
    results["bootstrap"][p] = {m: {"ci95": ci(store[p][m])} for m in ["pooled", "wq", "qonly"]}
    if p != "fp16":
        for m in ["pooled", "wq"]:
            fp, qp = np.array(store["fp16"][m]), np.array(store[p][m])
            diff = qp - fp
            ret = (qp - 0.5) / (fp - 0.5)
            results["bootstrap"][p][m]["diff_vs_fp16"] = {"mean": float(np.nanmean(diff)), "ci95": ci(diff)}
            results["bootstrap"][p][m]["retention"] = {"mean": float(np.nanmean(ret)), "ci95": ci(ret)}
        # H1 test: retention(wq) - retention(pooled)
        r_wq = (np.array(store[p]["wq"]) - .5) / (np.array(store["fp16"]["wq"]) - .5)
        r_po = (np.array(store[p]["pooled"]) - .5) / (np.array(store["fp16"]["pooled"]) - .5)
        results["bootstrap"][p]["H1_retention_gap_wq_minus_pooled"] = {"mean": float(np.nanmean(r_wq - r_po)), "ci95": ci(r_wq - r_po)}

## 8. Summary and save

In [ ]:
summ = pd.DataFrame({p: {"layer": results["main"][p]["layer"],
                         "greedy acc": greedy_acc.get(p),
                         "pooled AUROC": results["main"][p]["probe_pooled"],
                         "within-Q AUROC": results["main"][p]["probe_wq"],
                         "question-only AUROC": results["main"][p]["qonly_pooled"],
                         "logprob pooled": results["main"][p]["logprob_pooled"],
                         "logprob within-Q": results["main"][p]["logprob_wq"]} for p in PRECISIONS}).T
print(summ.round(3).to_string())
print("\nTransfer (pooled / within-Q):")
for k, v in results["transfer"].items(): print(f"  {k}: {v['pooled']:.3f} / {v['wq']:.3f}")
for p in PRECISIONS:
    if p != "fp16":
        print(p, "H1 retention gap (wq - pooled):", results["bootstrap"][p]["H1_retention_gap_wq_minus_pooled"])

summ.to_csv(f"{OUT_DIR}/summary.csv")
with open(f"{OUT_DIR}/results.json", "w") as f:
    json.dump(results, f, indent=1, default=float)
with zipfile.ZipFile(f"{OUT_DIR}/results.zip", "w") as z:
    for fn in ["results.json", "summary.csv", "answers.csv", "test_scores.csv"]:
        z.write(f"{OUT_DIR}/{fn}", fn)
print("Done. Download results.zip and send it back.")